# Train Crime-Event Prediction Model

Train the crime-event prediction model that powers the daily-options markets in the crimex app.
Trains a LightGBM regressor to predict the count of incidents in a (city, incident_type, horizon) window.
Exports a JSON tree-dump that the Node side (`lib/predictions/infrastructure/models/trained.ts`) can
evaluate without any ML runtime dependencies.

In [ ]:
%pip install -q lightgbm pandas numpy scipy requests scikit-learn matplotlib

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json
import math
import os
import random
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import poisson as scipy_poisson
from sklearn.metrics import log_loss as sk_log_loss
from sklearn.preprocessing import LabelEncoder

# ── Config — edit these before running ───────────────────────────────────────
ARCGIS_URL = "https://services2.arcgis.com/o1LYr96CpFkfsDJS/arcgis/rest/services/Crime_Map/FeatureServer/0"
LOOKBACK_DAYS = 730
HORIZONS_HOURS = [4, 8, 12, 24]
SUPABASE_URL = ""
SUPABASE_SERVICE_ROLE_KEY = ""
MODEL_ID = "trained-v1"
TIMEZONE = "America/Toronto"

# ── Data shape ──────────────────────────────────────────────────────────────
# 1h anchor step gives 6× more training samples than 6h. Slower, more accurate.
ANCHOR_STEP_HOURS = 1
MIN_INCIDENTS_PER_GROUP = 20

# ── Train/val/test split (calendar days from end of dataset) ─────────────────
TEST_DAYS = 60
VAL_DAYS  = 30  # carved out before test, used for early stopping

# ── LightGBM hyperparameters (per horizon — each model trained with these) ──
LGB_PARAMS = dict(
    objective         = "poisson",
    learning_rate     = 0.015,
    num_leaves        = 127,
    min_data_in_leaf  = 50,
    feature_fraction  = 0.85,
    bagging_fraction  = 0.85,
    bagging_freq      = 5,
    lambda_l2         = 1.0,
    max_depth         = -1,
    n_estimators      = 5000,
    n_jobs            = -1,
    verbose           = -1,
)
EARLY_STOPPING_ROUNDS = 200

# ── Kaggle Secrets fallback (no-op outside Kaggle) ──────────────────────────
try:
    from kaggle_secrets import UserSecretsClient  # type: ignore
    _ks = UserSecretsClient()
    SUPABASE_URL = SUPABASE_URL or _ks.get_secret("SUPABASE_URL")
    SUPABASE_SERVICE_ROLE_KEY = (
        SUPABASE_SERVICE_ROLE_KEY or _ks.get_secret("SUPABASE_SERVICE_ROLE_KEY")
    )
except Exception:
    pass

# Env-var fallback (Colab + local)
SUPABASE_URL = SUPABASE_URL or os.environ.get("SUPABASE_URL", "")
SUPABASE_SERVICE_ROLE_KEY = (
    SUPABASE_SERVICE_ROLE_KEY or os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "")
)

TZ = ZoneInfo(TIMEZONE)
print(f"Config: lookback={LOOKBACK_DAYS}d, horizons={HORIZONS_HOURS}, "
      f"anchor_step={ANCHOR_STEP_HOURS}h, min_per_group={MIN_INCIDENTS_PER_GROUP}")
print(f"Split:  test={TEST_DAYS}d, val={VAL_DAYS}d (carved before test)")
print(f"Model:  {LGB_PARAMS['objective']}, n_est={LGB_PARAMS['n_estimators']}, "
      f"lr={LGB_PARAMS['learning_rate']}, leaves={LGB_PARAMS['num_leaves']}, "
      f"early_stop={EARLY_STOPPING_ROUNDS}")
print(f"Supabase upload: {'enabled' if SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY else 'disabled'}")


In [ ]:
def fetch_arcgis(lookback_days: int) -> pd.DataFrame:
    """Paginated fetch from ArcGIS FeatureServer. Returns a flat DataFrame.

    Field names match the Halton Crime Map FeatureServer schema used by lib/arcgis.ts:
    OBJECTID, DATE (millis since epoch), CITY, DESCRIPTION, CASE_NO.
    """
    cutoff_ms = int(
        (datetime.now(timezone.utc).timestamp() - lookback_days * 86400) * 1000
    )
    records = []
    offset = 0
    page_size = 2000
    total_seen = 0
    total_kept = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "OBJECTID,DATE,CITY,DESCRIPTION,CASE_NO",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "orderByFields": "OBJECTID ASC",
        }
        resp = requests.get(f"{ARCGIS_URL}/query", params=params, timeout=60)
        resp.raise_for_status()
        geojson = resp.json()
        features = geojson.get("features", [])
        if not features:
            break

        for feat in features:
            total_seen += 1
            props = feat.get("properties") or feat.get("attributes", {})
            geom = feat.get("geometry") or {}

            date_val = props.get("DATE")
            if date_val is None:
                continue
            date_ms = int(date_val)
            if date_ms < cutoff_ms:
                continue

            coords = geom.get("coordinates", [None, None])
            records.append({
                "objectid":   props.get("OBJECTID"),
                "date_ms":    date_ms,
                "city":       str(props.get("CITY") or "unknown"),
                "description": str(props.get("DESCRIPTION") or "unknown"),
                "case_no":    props.get("CASE_NO"),
                "lng": coords[0] if coords and len(coords) > 1 else None,
                "lat": coords[1] if coords and len(coords) > 1 else None,
            })
            total_kept += 1

        if len(features) < page_size:
            break
        offset += page_size
        print(f"  fetched up to offset {offset} (kept {total_kept}/{total_seen}) ...")

    df = pd.DataFrame(records)
    print(f"fetch_arcgis: {len(df)} rows after cutoff filter (saw {total_seen} total)")
    if len(df) == 0:
        raise RuntimeError(
            "No rows returned. Check ARCGIS_URL is reachable and that the FeatureServer "
            "still uses the OBJECTID/DATE/CITY/DESCRIPTION/CASE_NO field names."
        )
    return df


df_raw = fetch_arcgis(LOOKBACK_DAYS)


In [ ]:
def _cyclical_arrays(values: np.ndarray, period: float):
    """Vectorized sin/cos of (2π · value / period)."""
    angle = 2 * math.pi * values / period
    return np.sin(angle), np.cos(angle)


def build_training_table(df: pd.DataFrame, horizons: list) -> pd.DataFrame:
    """Build feature rows for every (city, incident_type, anchor, horizon) combo.

    Vectorized: per-incident (dow, hour) computed once per group, then rolling
    counts are pure numpy slicing.
    """
    df = df.copy()
    df["incident_type"] = df["description"].str.upper().str.strip()

    anchor_step_ms = ANCHOR_STEP_HOURS * 3600 * 1000
    warmup_ms = 8 * 7 * 24 * 3600 * 1000  # 8 weeks

    global_min_ms = int(df["date_ms"].min())
    global_max_ms = int(df["date_ms"].max())
    warmup_cutoff_ms = global_min_ms + warmup_ms
    anchor_start_ms = (warmup_cutoff_ms // anchor_step_ms) * anchor_step_ms
    anchors_ms = np.arange(anchor_start_ms, global_max_ms, anchor_step_ms, dtype=np.int64)
    n_anchors = len(anchors_ms)
    if n_anchors == 0:
        raise RuntimeError("No anchors generated — dataset may be too short for the 8-week warmup.")

    # Anchor metadata (TZ-aware), computed once
    anchor_dts = [datetime.fromtimestamp(int(a) / 1000, tz=TZ) for a in anchors_ms]
    anchor_dow = np.array([d.weekday() for d in anchor_dts], dtype=np.int8)
    anchor_hour = np.array([d.hour for d in anchor_dts], dtype=np.int8)
    anchor_month = np.array([d.month for d in anchor_dts], dtype=np.int8)

    # Cyclical encodings (vectorized over anchors)
    dow_sin_arr,   dow_cos_arr   = _cyclical_arrays(anchor_dow.astype(float), 7)
    hour_sin_arr,  hour_cos_arr  = _cyclical_arrays(anchor_hour.astype(float), 24)
    # Month is 1..12 to match Node-side trained.ts evaluator
    month_sin_arr, month_cos_arr = _cyclical_arrays(anchor_month.astype(float), 12)

    # Window offsets
    h_ms  = lambda h: h * 3600 * 1000
    d_ms  = lambda d: d * 24 * 3600 * 1000
    w_ms  = lambda w: w * 7 * 24 * 3600 * 1000
    horizon_offsets = np.array([h_ms(h) for h in horizons], dtype=np.int64)

    # Filter sparse groups
    group_keys = df.groupby(["city", "incident_type"]).size()
    keep_keys = set(group_keys[group_keys >= MIN_INCIDENTS_PER_GROUP].index)
    df = df.set_index(["city", "incident_type"]).loc[list(keep_keys)].reset_index()
    n_groups = len(keep_keys)
    print(f"build_training_table: {n_groups} groups (filtered to >= {MIN_INCIDENTS_PER_GROUP} incidents)")

    rows = []
    progress_every = max(1, n_groups // 20)

    for gi, ((city, itype), group) in enumerate(df.groupby(["city", "incident_type"])):
        ts = np.sort(group["date_ms"].values.astype(np.int64))
        # Per-incident (dow, hour) once — was the prior bottleneck
        dts = [datetime.fromtimestamp(int(t) / 1000, tz=TZ) for t in ts]
        ts_dow  = np.array([d.weekday() for d in dts], dtype=np.int8)
        ts_hour = np.array([d.hour     for d in dts], dtype=np.int8)

        for ai in range(n_anchors):
            a = int(anchors_ms[ai])
            adow  = int(anchor_dow[ai])
            ahour = int(anchor_hour[ai])

            ms_24h = a - d_ms(1)
            ms_7d  = a - d_ms(7)
            ms_4w  = a - w_ms(4)
            ms_8w  = a - w_ms(8)

            # All counts use semi-open [lo, hi)
            i_a   = int(np.searchsorted(ts, a, side="left"))
            i_24h = int(np.searchsorted(ts, ms_24h, side="left"))
            i_7d  = int(np.searchsorted(ts, ms_7d,  side="left"))
            i_4w  = int(np.searchsorted(ts, ms_4w,  side="left"))
            i_8w  = int(np.searchsorted(ts, ms_8w,  side="left"))

            c24h = i_a - i_24h
            c7d  = i_a - i_7d
            c4w  = int(((ts_dow[i_4w:i_a] == adow) & (ts_hour[i_4w:i_a] == ahour)).sum())
            c8w  = int(((ts_dow[i_8w:i_a] == adow) & (ts_hour[i_8w:i_a] == ahour)).sum())

            # Targets per horizon
            for hi_, horizon_h in enumerate(horizons):
                end_ms = a + int(horizon_offsets[hi_])
                i_end  = int(np.searchsorted(ts, end_ms, side="left"))
                target = i_end - i_a

                rows.append((
                    float(dow_sin_arr[ai]),   float(dow_cos_arr[ai]),
                    float(hour_sin_arr[ai]),  float(hour_cos_arr[ai]),
                    float(month_sin_arr[ai]), float(month_cos_arr[ai]),
                    c4w, c8w, c24h, c7d,
                    horizon_h,
                    itype, city,
                    target,
                    a,
                ))

        if (gi + 1) % progress_every == 0 or gi + 1 == n_groups:
            print(f"  group {gi+1}/{n_groups}: rows so far = {len(rows)}")

    cols = [
        "dow_sin", "dow_cos", "hour_sin", "hour_cos", "month_sin", "month_cos",
        "count_4w_same_dow_hour", "count_8w_same_dow_hour",
        "count_24h", "count_7d",
        "horizon_hours",
        "incident_type", "city",
        "target_count",
        "anchor_ms",
    ]
    out = pd.DataFrame.from_records(rows, columns=cols)
    print(f"build_training_table: {len(out)} rows total")
    return out


df_train_raw = build_training_table(df_raw, HORIZONS_HOURS)


In [ ]:
# ── Time-based 3-way split ───────────────────────────────────────────────────
max_anchor = int(df_train_raw["anchor_ms"].max())
test_cutoff_ms = max_anchor - TEST_DAYS * 24 * 3600 * 1000
val_cutoff_ms  = test_cutoff_ms - VAL_DAYS * 24 * 3600 * 1000

train_mask = df_train_raw["anchor_ms"] <  val_cutoff_ms
val_mask   = (df_train_raw["anchor_ms"] >= val_cutoff_ms) & (df_train_raw["anchor_ms"] < test_cutoff_ms)
test_mask  = df_train_raw["anchor_ms"] >= test_cutoff_ms

print(f"Train rows: {train_mask.sum():>9}")
print(f"Val   rows: {val_mask.sum():>9}")
print(f"Test  rows: {test_mask.sum():>9}")

# ── Label-encode categoricals once across the entire dataset ─────────────────
le_type = LabelEncoder()
le_city = LabelEncoder()
le_type.fit(df_train_raw["incident_type"].unique())
le_city.fit(df_train_raw["city"].unique())
df_train_raw["type_id"] = le_type.transform(df_train_raw["incident_type"])
df_train_raw["city_id"] = le_city.transform(df_train_raw["city"])

type_encoder_map = {cls: int(idx) for idx, cls in enumerate(le_type.classes_)}
city_encoder_map = {cls: int(idx) for idx, cls in enumerate(le_city.classes_)}

# Per-horizon model — horizon_hours is implicit, NOT a feature.
FEATURE_NAMES = [
    "dow_sin", "dow_cos",
    "hour_sin", "hour_cos",
    "month_sin", "month_cos",
    "count_4w_same_dow_hour",
    "count_8w_same_dow_hour",
    "count_24h",
    "count_7d",
    "type_id",
    "city_id",
]
CATEGORICAL_FEATURES = ["type_id", "city_id"]

def _slice(df, mask, horizon):
    sub = df[mask & (df["horizon_hours"] == horizon)]
    X = sub[FEATURE_NAMES].copy()
    # LightGBM categorical mode requires int dtype for these columns
    for c in CATEGORICAL_FEATURES:
        X[c] = X[c].astype("int32")
    y = sub["target_count"].values.astype(float)
    return X, y, sub

# ── Train one LGBMRegressor per horizon ──────────────────────────────────────
models_by_horizon = {}
for horizon in HORIZONS_HOURS:
    print(f"\n────── horizon = {horizon}h ──────")
    X_tr, y_tr, _ = _slice(df_train_raw, train_mask, horizon)
    X_va, y_va, _ = _slice(df_train_raw, val_mask,   horizon)
    print(f"  train: {len(X_tr):>8}   val: {len(X_va):>8}   "
          f"mean target = {y_tr.mean():.3f}   max = {int(y_tr.max())}")

    if len(X_tr) == 0 or len(X_va) == 0:
        print(f"  SKIP — empty split for horizon {horizon}h")
        continue

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="mae",
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    val_mae = model.best_score_["valid_0"]["l1"]
    print(f"  best iter: {model.best_iteration_}   val MAE: {val_mae:.4f}")
    models_by_horizon[horizon] = model


In [ ]:
# ── Per-horizon calibration evaluation on the held-out test set ──────────────
per_horizon_metrics = {}

for horizon, model in models_by_horizon.items():
    X_te, y_te, _ = _slice(df_train_raw, test_mask, horizon)
    if len(X_te) == 0:
        print(f"horizon={horizon}h: empty test set, skipping")
        continue

    y_pred = model.predict(X_te)
    y_pred = np.clip(y_pred, 1e-6, None)
    mae = float(np.mean(np.abs(y_pred - y_te)))

    thresholds = np.maximum(np.round(y_pred), 1).astype(int)
    p_exceed = np.array([
        1.0 - scipy_poisson.cdf(t - 1, mu=mu)
        for t, mu in zip(thresholds, y_pred)
    ])
    y_binary = (y_te >= thresholds).astype(int)

    brier = float(np.mean((p_exceed - y_binary) ** 2))
    if len(np.unique(y_binary)) < 2:
        log_loss_val = float("nan")
    else:
        p_clipped = np.clip(p_exceed, 1e-7, 1 - 1e-7)
        log_loss_val = float(sk_log_loss(y_binary, p_clipped, labels=[0, 1]))

    per_horizon_metrics[horizon] = {
        "mae": mae, "brier": brier, "log_loss": log_loss_val,
        "n_test": int(len(y_te)),
        "positive_rate": float(y_binary.mean()),
    }
    print(f"horizon={horizon:>2}h  MAE={mae:.4f}  Brier={brier:.4f}  "
          f"LogLoss={log_loss_val:.4f}  +rate={100*y_binary.mean():.1f}%  n={len(y_te)}")

# ── Pooled reliability diagram (across horizons) ─────────────────────────────
all_p, all_y = [], []
for horizon, model in models_by_horizon.items():
    X_te, y_te, _ = _slice(df_train_raw, test_mask, horizon)
    if len(X_te) == 0:
        continue
    yp = np.clip(model.predict(X_te), 1e-6, None)
    th = np.maximum(np.round(yp), 1).astype(int)
    pe = np.array([1.0 - scipy_poisson.cdf(t - 1, mu=mu) for t, mu in zip(th, yp)])
    all_p.append(pe)
    all_y.append((y_te >= th).astype(int))
all_p = np.concatenate(all_p) if all_p else np.array([])
all_y = np.concatenate(all_y) if all_y else np.array([])

if len(all_p):
    n_bins = 10
    edges = np.linspace(0, 1, n_bins + 1)
    fp, mp = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (all_p >= lo) & (all_p < hi)
        if m.sum() == 0:
            continue
        fp.append(all_y[m].mean())
        mp.append(all_p[m].mean())
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    if mp:
        ax.plot(mp, fp, "o-", label="Model (pooled)")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title("Reliability diagram (all horizons)")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Export one snapshot per horizon ──────────────────────────────────────────
def _walk_tree(node, x):
    """Byte-for-byte mirror of the JS evaluator in lib/predictions/infrastructure/models/trained.ts.

    Handles both numeric splits (decision_type "<=") and categorical splits
    (decision_type "==", threshold encoded as "v1||v2||..." set of int ids).
    """
    while True:
        sf = node.get("split_feature")
        if sf is None:
            return float(node["leaf_value"])
        thr = node["threshold"]
        dt = node.get("decision_type", "<=")
        v = x[sf]
        if dt == "==":
            # Categorical: threshold is the set of category ids that go LEFT.
            target = int(v)
            if isinstance(thr, str):
                go_left = any(int(p) == target for p in thr.split("||"))
            else:
                go_left = target == int(thr)
        else:
            go_left = v <= thr
        node = node["left_child"] if go_left else node["right_child"]


def _build_snapshot(model, horizon, metrics):
    booster = model.booster_
    dump = booster.dump_model()
    trees = dump["tree_info"]

    # Reverse-engineer init_score using one training sample.
    X_tr, _, _ = _slice(df_train_raw, train_mask, horizon)
    sample = X_tr.iloc[0].values.astype(float)
    raw_score = float(booster.predict(sample.reshape(1, -1), raw_score=True)[0])
    sum_leaves = sum(_walk_tree(t["tree_structure"], sample) for t in trees)
    init_score = raw_score - sum_leaves

    # Self-test: walker + init_score must reproduce booster predictions
    random.seed(horizon)
    n_check = min(50, len(X_tr))
    idxs = random.sample(range(len(X_tr)), n_check)
    max_err = 0.0
    for idx in idxs:
        s = X_tr.iloc[idx].values.astype(float)
        expected = float(model.predict(s.reshape(1, -1))[0])
        got = math.exp(init_score + sum(_walk_tree(t["tree_structure"], s) for t in trees))
        err = abs(expected - got) / max(abs(expected), 1e-6)
        max_err = max(max_err, err)
    if max_err > 1e-3:
        raise AssertionError(
            f"horizon={horizon}h: tree walker disagrees with LightGBM (max rel err {max_err:.2e}). "
            f"Investigate before uploading."
        )

    snapshot = {
        "format":           "lightgbm-treedump-v1",
        "feature_names":    FEATURE_NAMES,
        "categorical": {
            "incident_type": type_encoder_map,
            "city":          city_encoder_map,
        },
        "trees":            trees,
        "init_score":       init_score,
        "objective":        LGB_PARAMS["objective"],
        "trained_at":       datetime.now(timezone.utc).isoformat(),
        "training_samples": int(len(X_tr)),
        "best_iteration":   int(model.best_iteration_),
        "metrics": {
            "mae":      metrics["mae"],
            "brier":    metrics["brier"],
            "log_loss": metrics["log_loss"] if not math.isnan(metrics["log_loss"]) else None,
            "n_test":   metrics["n_test"],
        },
        "horizon_hours": horizon,
    }
    return snapshot, max_err


snapshots_by_horizon = {}
for horizon, model in models_by_horizon.items():
    metrics = per_horizon_metrics.get(horizon)
    if metrics is None:
        print(f"horizon={horizon}h: skipped — no test metrics")
        continue
    snap, max_err = _build_snapshot(model, horizon, metrics)
    snapshots_by_horizon[horizon] = snap
    snap_size_mb = len(json.dumps(snap).encode("utf-8")) / 1_048_576
    print(f"horizon={horizon:>2}h  trees={len(snap['trees']):>4}  "
          f"init={snap['init_score']:+.4f}  walker_err={max_err:.2e}  size={snap_size_mb:.2f} MB")

# Save individual JSON files for inspection
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for horizon, snap in snapshots_by_horizon.items():
    p = os.path.join(out_dir, f"snapshot_{horizon}h.json")
    with open(p, "w") as f:
        json.dump(snap, f)
    print(f"  saved {p}")


In [ ]:
# ── Upload one row per horizon to prediction_model_snapshots ─────────────────
# Per-snapshot POSTs (snapshots can be 20+ MB each — batched upload times out).
if SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY and snapshots_by_horizon:
    endpoint = f"{SUPABASE_URL.rstrip('/')}/rest/v1/prediction_model_snapshots"
    headers = {
        "Authorization":  f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
        "apikey":         SUPABASE_SERVICE_ROLE_KEY,
        "Content-Type":   "application/json",
        "Prefer":         "resolution=merge-duplicates",
    }
    ok_count = 0
    for h, snap in snapshots_by_horizon.items():
        row = [{"model_id": MODEL_ID, "horizon_hours": h, "state": snap}]
        resp = requests.post(endpoint, headers=headers, json=row, timeout=300)
        size_mb = len(json.dumps(row)) / (1024 * 1024)
        print(f"horizon={h}h  size={size_mb:.2f} MB  status={resp.status_code}")
        if not resp.ok:
            print(resp.text[:500])
        else:
            ok_count += 1
    print(f"Uploaded {ok_count}/{len(snapshots_by_horizon)} snapshots.")
elif not snapshots_by_horizon:
    print("No snapshots produced — skipping upload.")
else:
    print("SUPABASE_URL / SUPABASE_SERVICE_ROLE_KEY not set — skipping upload.")


## How to run

### Kaggle

1. Go to **kaggle.com → Code → New Notebook**, then **File → Import Notebook** and upload `train_crime_model.ipynb`.
2. Add your Supabase credentials via **Add-ons → Secrets** (name them `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`).
   Read them in the config cell with `os.environ.get("SUPABASE_URL", "")`.
3. Set **Accelerator → None** (CPU is fine for LightGBM) and click **Run All**.
4. The output file will be written to `/kaggle/working/snapshot.json` and auto-persisted as a Kaggle output.

### Google Colab

1. Open **colab.research.google.com**, create a new notebook, and paste the cells from this file (or use **File → Upload notebook**).
2. Set `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY` directly in the config cell, or read them from Colab Secrets (`userdata.get(...)`).
3. Click **Runtime → Run all**.
4. The snapshot will be saved to `./snapshot.json` in the Colab working directory; download it from the Files panel.

### Node side

Once you click **Upload** (or the Supabase POST completes), the JSON snapshot is stored in the
`prediction_model_snapshots` table keyed by `(model_id, horizon_hours)`.  
The Node side reads it via `getModelStateSnapshot` in
`lib/predictions/infrastructure/supabaseRepos.ts` and the tree-walking evaluator in
`lib/predictions/infrastructure/models/trained.ts` scores each prediction without any ML runtime dependency.